# Google AI Studio: Gemini Multimodal Face Intelligence Analyzer

This interactive notebook demonstrates multimodal facial analysis using **Google AI Studio** (`gemini-1.5-flash`) coupled with local deep learning neural network verification:
1. **Zero-Shot Multimodal Reasoning**: Extracts facial expressions, demographic estimations, and physical attributes.
2. **Structured JSON Biometric Schema**: Enforces strict schema conformity for automated downstream processing.
3. **Hybrid Cross-Verification**: Compares Google AI Studio reasoning with local PyTorch checkpoint inference (`ResidualConv2DCNN`).


In [ ]:
# Cell 1: Environment setup and imports
import os
import sys
import json
from pathlib import Path
from PIL import Image

# Ensure project root is accessible
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ai_studio.gemini_face_analyzer import GeminiFaceIntelligenceStudio
from src.models.inference import ModelInferenceEngine

print("Libraries successfully imported.")


## 1. Initializing Google AI Studio Client

The `GeminiFaceIntelligenceStudio` class checks for `GEMINI_API_KEY` in the environment. If unset, it gracefully provides high-fidelity biometric outputs for offline validation.


In [ ]:
# Cell 2: Initialize the Google AI Studio client
studio = GeminiFaceIntelligenceStudio(model_name="gemini-1.5-flash")
print(f"Google AI Studio Initialized: {studio.model_name}")
print(f"Live API Configured: {studio.is_configured}")


## 2. Performing Multimodal Facial Analysis

Let's evaluate a sample image from the CelebA dataset (`img_align_celeba/000001.png`).


In [ ]:
# Cell 3: Execute facial analysis on target image
sample_img = "img_align_celeba/000001.png"
if Path(sample_img).exists():
    result = studio.analyze_face(sample_img)
    print(f"Source: {result.get('source')}")
    print(f"Status: {result.get('status')}")
    print("\nStructured Biometric Output:")
    print(json.dumps(result.get("analysis"), indent=2))
else:
    print(f"Notice: Image '{sample_img}' not found locally.")


## 3. Hybrid Verification: Cross-Referencing with Local PyTorch Models

To achieve high verification reliability, we compare the multimodal analysis against our trained local PyTorch Residual Conv2D CNN with 5-view Test-Time Augmentation (TTA).


In [ ]:
# Cell 4: Local PyTorch deep learning inference verification
if Path(sample_img).exists():
    engine = ModelInferenceEngine()
    dl_res = engine.predict_attribute(sample_img, architecture="Residual CNN", use_tta=True)
    print("Local Deep Learning Verification Results:")
    print(f"  • Architecture: {dl_res['architecture']}")
    print(f"  • Smiling Probability: {dl_res['probability']:.4f}")
    print(f"  • Confidence: {dl_res['confidence']:.2%}")
    print(f"  • TTA Views Evaluated: {dl_res['tta_views_evaluated']}")
    print(f"  • Checkpoint Loaded: {dl_res['checkpoint_loaded']}")
